<h3>Read CT, convert to pointcloud, filter vertices and plot result</h3>

In [174]:
import nibabel as nib
import numpy as np

nii_file = "/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/7 Blanco AiCE 3.000 segmentation.nii"
img = nib.load(nii_file)

data = img.get_fdata()
affine = img.affine

print("Data shape:", data.shape)

# 1) Boolean mask for label==1
mask = (data == 1)

Data shape: (512, 512, 152)


In [175]:
# 2) For each (x, z) column, find the highest y where mask is True
# Build an array of y indices with -1 where not occupied, then take max along axis=1 (the y-axis)
Y = np.arange(data.shape[1], 0, -1)[None, :, None]                   # shape (1, Y, 1)
y_idx = np.where(mask, Y, -1)                                 # shape (X, Y, Z)
top_y = y_idx.max(axis=1)                                     # shape (X, Z), -1 where no label in column

# 3) Collect (x, y_top, z) triples where there is at least one occupied voxel
xs, zs = np.where(top_y >= 0)                                 # arrays of indices for valid columns
ys = top_y[xs, zs]
voxel_coords = np.column_stack([xs, ys, zs])                  # shape (N, 3) in (x, y, z) voxel space

# 4) Convert voxel (i,j,k) to world coordinates using the affine
hom = np.c_[voxel_coords, np.ones(len(voxel_coords))]
world_coords = (hom @ affine.T)[:, :3]

print(f"Selected {world_coords.shape[0]} topmost points (one per (x,z) column).")

Selected 71196 topmost points (one per (x,z) column).


In [184]:
select_coords = world_coords[world_coords[:, 2] < -1100]
select_coords = select_coords[select_coords[:, 1] < -30]
select_coords /= 10.  # mm to cm
select_coords -= np.mean(select_coords, axis=0)
select_coords = select_coords[:, [0, 2, 1]]
print(f"Kept {select_coords.shape[0]} points out of {world_coords.shape[0]}")

Kept 26299 points out of 71196


In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=[go.Scatter3d(
    x=select_coords[:, 0],
    y=select_coords[:, 1],
    z=select_coords[:, 2],
    mode="markers",
    marker=dict(
        size=1,
        opacity=0.5,
        color="royalblue"
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    title="Segmentation Label 1 Point Cloud",
    width=1200,
    height=800
)

fig.show()

<h3>If satisfied, save pointcloud and pre-process in MeshLab</h3>

In [186]:
import open3d as o3d
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(select_coords)
o3d.io.write_point_cloud("/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/ct_scan2.ply", pcd)

True

<h3>Read pre-processed file and store as data point</h3>

In [188]:
def orient_normals_negative_zaxis(normals):
    normals = np.asarray(normals)
    mask = normals[:, 2] > 0
    normals[mask] *= -1
    return o3d.cuda.pybind.utility.Vector3dVector(normals)

In [189]:
mesh = o3d.io.read_triangle_mesh("/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/ct_scan2_preprocessed.ply")
mesh.vertex_normals = orient_normals_negative_zaxis(mesh.vertex_normals)

mesh_data = {
    "input_points": torch.as_tensor(np.asarray(mesh.vertices)),
    "input_normals": torch.as_tensor(np.asarray(mesh.vertex_normals)),
    "input_faces": torch.as_tensor(np.asarray(mesh.triangles, dtype=np.int64))
}

torch.save(mesh_data, "/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/ct_scan2.pt")
o3d.io.write_triangle_mesh("/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/ct_scan2_normals.ply", mesh)

True

<h3>Plot CT and RGBD together</h3>

In [190]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def add_markers(pts, size=2, opacity=.8, color="red", colorscale=None, showscale=False, cmin=None, cmax=None):
    return go.Scatter3d(
        x = pts[:, 0], y = pts[:, 1], z = -pts[:, 2], mode = "markers",
        marker = {
            "size": size,
            "opacity": opacity,
            "color": color,
            "colorscale": colorscale,
            "showscale": showscale,
            "cmin": cmin,
            "cmax": cmax
        })

def add_lines(pts_a, pts_b, width=2, color="black"):
    x, y, z = [], [], []
    for p, q in zip(pts_a, pts_b):
        x += [p[0], q[0], None]
        y += [p[1], q[1], None]
        z += [-p[2], -q[2], None]
    return go.Scatter3d(
        x = x, y = y, z = z, mode = "lines",
        line = {
            "color": color,
            "width": width
        })

def plot_data(data):
    fig = go.Figure(data=data)
    fig.update_layout(
        width = 800,
        height = 800,
        template = "plotly_white",
        scene = {"xaxis": {"visible": False}, "yaxis": {"visible": False}, "zaxis": {"visible": False}}, 
        scene_camera = {"eye": {"x": 0.0, "y": 1.0, "z": 1.5}}
    )
    fig.show()

def plot_abdomen(pts, marks=None):
    data = []
    data.append(add_markers(pts, color=-pts[:, 2], opacity=1., colorscale="Viridis"))
    if marks is not None:
        data.append(add_markers(marks, size=10, opacity=1.))
    plot_data(data)

def plot_abdomen_with_errors(pts, errors, pts_a=None, marks=None):
    data = []
    data.append(add_markers(pts, color=errors, opacity=1., colorscale="RdBu_r", cmin=0, cmax=3))
    if pts_a is not None:
        data.append(add_markers(pts_a, color=-pts_a[:, 2], opacity=1., colorscale="Greens_r"))
    if marks is not None:
        data.append(add_markers(marks, size=10, opacity=1.))
    plot_data(data)

def plot_abdomens(pts_a, pts_b, marks_a=None, marks_b=None):
    data = []
    data.append(add_markers(pts_a, size=1, opacity=.5, color=-pts_a[:, 2], colorscale="Reds"))
    data.append(add_markers(pts_b, size=1, opacity=.2, color=-pts_b[:, 2], colorscale="Greens"))
    if marks_a is not None:
        data.append(add_markers(marks_a, size=5, opacity=1., color="black"))
    if marks_b is not None:
        data.append(add_markers(marks_b, size=5, opacity=1., color="black"))
    if marks_a is not None and marks_b is not None:
        data.append(add_lines(marks_a, marks_b))
    plot_data(data)

def plot_landmarks(pts, anns_a, anns_b, pred_a, pred_b):
    data = []
    data.append(add_markers(pts, color=-pts[:, 2], opacity=.5, colorscale="Viridis"))
    # for b, a in zip(before, after):
    #     data.append(add_lines(np.expand_dims(b, axis=0), np.expand_dims(a, axis=0), color="green"))
    # for b, a in zip(before, pred):
    #     data.append(add_lines(np.expand_dims(b, axis=0), np.expand_dims(a, axis=0), color="red"))
    data.append(add_lines(anns_a, anns_b, color="green"))
    data.append(add_lines(pred_a, pred_b, color="red"))
    plot_data(data)

def plots_ct_scans(depth_pts_a, depth_pts_b, ct_pts_a, ct_pts_b, depth_marks_a, depth_marks_b, ct_marks_a, ct_marks_b):
    fig = make_subplots(rows=1, cols=3, specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.04)
    # Depth plot
    fig.add_trace(add_markers(depth_pts_a, size=2, opacity=1., color=-depth_pts_a[:, 2], colorscale="Reds"), row=1, col=1)
    fig.add_trace(add_markers(depth_pts_b, size=2, opacity=.5, color=-depth_pts_b[:, 2], colorscale="Greens"), row=1, col=1)
    # CT plot
    fig.add_trace(add_markers(ct_pts_a, size=2, opacity=1., color=-ct_pts_a[:, 2], colorscale="Reds"), row=1, col=2)
    fig.add_trace(add_markers(ct_pts_b, size=2, opacity=.5, color=-ct_pts_b[:, 2], colorscale="Greens"), row=1, col=2)
    # Landmarks plot
    fig.add_trace(add_markers(ct_pts_a, color=-ct_pts_a[:, 2], opacity=1., colorscale="Viridis"), row=1, col=3)
    fig.add_trace(add_lines(depth_marks_a, depth_marks_b, color="Green"), row=1, col=3)
    fig.add_trace(add_lines(ct_marks_a, ct_marks_b, color="Red"), row=1, col=3)
    scene_axes = {"xaxis": {"visible": False}, "yaxis": {"visible": False}, "zaxis": {"visible": False}}
    cam = {"eye": {"x": 1.2, "y": -1.6, "z": 1.2}}
    fig.update_layout(
        width=1200, height=400, template="plotly_white",
        scene=scene_axes, scene2=scene_axes, scene3=scene_axes,
        scene_camera=cam, scene2_camera=cam, scene3_camera=cam,
        margin=dict(l=0, r=0, t=0, b=0)
    )
    fig.show()

In [75]:
def rmse(pts_a, pts_b):
    return np.linalg.norm(pts_a - pts_b, axis=-1)

def mae(pts_a, pts_b):
    return np.sum(np.abs(pts_a - pts_b), axis=-1) / 3.

def magn(pts_a, pts_b):
    magn_a = np.linalg.norm(pts_a, axis=-1).clip(min=1e-8)
    magn_b = np.linalg.norm(pts_b, axis=-1)
    return np.abs(magn_a - magn_b) / magn_a * 100

def ang(pts_a, pts_b):
    dot = np.sum(pts_b * pts_a, axis=-1)
    denom = np.linalg.norm(pts_b, axis=-1).clip(min=1e-8) * np.linalg.norm(pts_a, axis=-1).clip(min=1e-8)
    cos = np.clip(dot / denom, a_min=-1., a_max=1.)
    return np.arccos(cos) * 180. / np.pi

In [191]:
from scipy.spatial import cKDTree

def transform_pts(pts, transform):
    ones = np.ones((pts.shape[0], 1))
    pts = np.hstack([pts, ones])
    return (pts @ transform.T)[:, :3]

transform = np.array(
    [[-0.98562271,  0.03463216, -0.16537376, -0.61375112],
     [-0.05613675, -0.99028858,  0.12718957, -3.8648776 ],
     [-0.15936289,  0.13464447,  0.97799506, -1.56822681],
     [ 0., 0., 0., 1.]]
)

transform2 = np.array(
    [[0.998644, 0.0113415, -0.050811, 0.120569],
    [0.0047647, -0.991797, -0.127734, 5.40617],
    [0.0518429, -0.127318, 0.990506, -1.32093],
    [0, 0, 0, 1]])

ct_mesh_input = o3d.io.read_triangle_mesh("/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/ct_scan2_preprocessed.ply")
ct_mesh_pred = o3d.io.read_triangle_mesh("/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/ct_scan2_pred.ply")
ct_pts_input = np.asarray(ct_mesh_input.vertices)
ct_pts_pred = np.asarray(ct_mesh_pred.vertices)

depth_data = torch.load("/data/Predict-Pneumoperitoneum_LaB-GATr/dataset/raw/24_12_CT+.pt", weights_only=False)
# depth_pts_input = o3d.io.read_triangle_mesh("/data/Predict-Pneumoperitoneum_LaB-GATr/ct_scans/24_12_CT+/start_0007.ply")
# depth_pts_input = transform_pts(np.asarray(depth_pts_input.vertices), transform2)
depth_pts_input = transform_pts(depth_data["input_points"].numpy() * 100, transform)
depth_pts_target = transform_pts(depth_data["target_points"].numpy() * 100, transform)
depth_marks_input = transform_pts(depth_data["annotations_start"].numpy() * 100, transform)
depth_marks_target = transform_pts(depth_data["annotations_end"].numpy() * 100, transform)
depth_marks_target_delta = depth_marks_target - depth_marks_input

# Calculate closest points for landmarks on CT
tree = cKDTree(ct_pts_input)
deltas, ct_marks_idxs = tree.query(depth_marks_input)
ct_marks_input = ct_pts_input[ct_marks_idxs]
ct_marks_pred = ct_pts_pred[ct_marks_idxs]
ct_marks_pred_delta = ct_marks_pred - ct_marks_input
print(f"Mean dist landmarks depth -> CT: {np.mean(deltas):.2f}")

rmse_mean = np.mean(rmse(ct_marks_pred_delta, depth_marks_target_delta))
mae_mean = np.mean(mae(ct_marks_pred_delta, depth_marks_target_delta))
magn_mean = np.mean(magn(ct_marks_pred_delta, depth_marks_target_delta))
ang_mean = np.mean(ang(ct_marks_pred_delta, depth_marks_target_delta))

print(f"RMSE: {rmse_mean:.2f}, MAE: {mae_mean:.2f}, MAGN: {magn_mean:.2f}, ANG: {ang_mean:.2f}")

[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
Mean dist landmarks depth -> CT: 0.76
RMSE: 2.21, MAE: 1.12, MAGN: 45.44, ANG: 15.59


In [ ]:
# plot_landmarks(ct_pts_input, depth_marks_input, depth_marks_target, ct_marks_input, ct_marks_pred)
plots_ct_scans(depth_pts_input, depth_pts_target, ct_pts_input, ct_pts_pred, depth_marks_input, depth_marks_target, ct_marks_input, ct_marks_pred)

In [ ]:
plot_abdomens(depth_pts_input, ct_pts_input)